# Data Visualization — Quiz (Lectures 1–7)

**Time:** 90 minutes 
**Tools:** Plotly only (`plotly.express` first, then `update_traces` / `update_layout` / `update_xaxes` / `update_yaxes`)

**Your task:** answer each analytical question with **one polished, explanatory visualization**. For every question **you decide which chart type fits best** — and justify the choice in one or two sentences.

**Every figure must have:**
1. An **insight title** that states the finding, not just the variables
2. `template='plotly_white'` and a decluttered layout (no unnecessary gridlines, legends or axis titles)
3. A **CVD-safe palette** — no red/green; grey for context + one accent colour where it helps
4. Colour that **encodes meaning**, not decoration
5. Direct labels or annotations where they help the reader

**Chart types covered in the course** — choose from these:
line chart · sorted bar chart · stacked / 100% stacked bar chart · scatter plot · bubble chart · histogram · box plot · violin plot · choropleth map · treemap

> 💡 Each question has one chart type that fits it best, and the questions together cover most of the list above. If two of your answers use the same chart type, check the question again.


**Submit:** this notebook pushed to your personal repo as `quiz/dataviz_quiz.ipynb`

## The dataset — World Population 1970–2022

**Source:** [World Population Dataset](https://www.kaggle.com/datasets/iamsouravbanerjee/world-population-dataset) on Kaggle, compiled from Worldometer (UN World Population Prospects estimates). **234 rows × 16 columns — one row per country or territory, no missing values.**

| Column | Type | Description |
|---|---|---|
| `Country`, `Capital` | categorical | Country / territory and its capital |
| `Continent` | categorical | Africa, Asia, Europe, North America, Oceania, South America |
| `CCA3` | spatial | ISO-3 country code — use with `locationmode='ISO-3'` |
| `1970 Population` … `2022 Population` | temporal | Population in 1970, 1980, 1990, 2000, 2010, 2015, 2020, 2022 |
| `Area` | numerical | Land area in km² |
| `Density` | numerical | People per km² (2022) |
| `World share (%)` | numerical | Share of world population in 2022 |
| `Rank` | numerical | Rank by 2022 population |

**The next setup cell also gives you `df_long` — the same data in **tidy (long) format** with one row per country per year. Use `df` for one-year questions and `df_long` for questions about change over time**

In [8]:
import pandas as pd
import plotly.express as px
import numpy as np

df = pd.read_csv('world_population.csv')

# Shorter column names
df = df.rename(columns={'Country/Territory': 'Country', 'Area (km²)': 'Area',
                        'Density (per km²)': 'Density', 'World Population Percentage': 'World share (%)'})

# Tidy (long) version: one row per country per year
year_cols = [c for c in df.columns if c.endswith('Population')]
df_long = df.melt(id_vars=['CCA3', 'Country', 'Continent'], value_vars=year_cols,
                  var_name='Year', value_name='Population')

df_long['Year'] = df_long['Year'].str[:4].astype(int)     # '2022 Population' → 2022

print(df.shape, df_long.shape)


(234, 16) (1872, 5)


---
## Q1 — Which continent is growing fastest?

How has the population of each continent **grown since 1970, relative to its 1970 size** — and which continent is growing fastest?


**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Line chart** with a full continent colour palette. Time series → lines; indexing to 1970 = 100 makes relative growth comparable. Every continent keeps its own colour so trajectories stay identifiable; Africa is drawn thicker and labelled as the fastest grower.

In [9]:
# Q1 — WHY a LINE chart?
# Population change over time is a continuous trend → a line encodes continuity.
# Indexing to 1970=100 makes continents comparable by *relative* growth, not raw size.
# Colour = continent identity; Africa is thicker because it is the fastest-growing story.

CONT = {
    'Africa': '#FF6B35', 'Asia': '#4C78A8', 'Europe': '#F2C14E',
    'North America': '#9B5DE5', 'South America': '#00BBF9', 'Oceania': '#00F5D4',
}

cont = df_long.groupby(['Continent', 'Year'], as_index=False)['Population'].sum()
base = cont.loc[cont['Year'] == 1970].set_index('Continent')['Population']
cont['Growth_index'] = cont.apply(lambda r: r['Population'] / base[r['Continent']] * 100, axis=1)
fastest = cont.loc[cont['Year'] == 2022].sort_values('Growth_index', ascending=False).iloc[0]

fig = px.line(
    cont, x='Year', y='Growth_index', color='Continent',
    color_discrete_map=CONT, markers=True,
    labels={'Growth_index': 'Population index (1970 = 100)', 'Year': ''},
    template='plotly_white', height=500,
)
fig.update_traces(line=dict(width=2.5), marker=dict(size=7))
fig.update_traces(selector=dict(name=fastest['Continent']), line=dict(width=5), marker=dict(size=10))

for cname in cont['Continent'].unique():
    end = cont[(cont['Continent'] == cname) & (cont['Year'] == 2022)].iloc[0]
    fig.add_annotation(
        x=end['Year'], y=end['Growth_index'],
        text=f"{cname} {end['Growth_index']:.0f}",
        showarrow=False, xanchor='left', xshift=8,
        font=dict(color=CONT[cname], size=11 if cname != fastest['Continent'] else 13, family='Arial'),
    )

fig.update_layout(
    title=f"Africa nearly quadrupled since 1970 (index {fastest['Growth_index']:.0f}) — every continent told in its own colour",
    font=dict(family='Arial', size=13), showlegend=False,
    yaxis=dict(gridcolor='#F0F0F0', title='Index (1970 = 100)', zeroline=False),
    xaxis=dict(showgrid=False), plot_bgcolor='#FAFBFF',
    margin=dict(l=60, r=130, t=70, b=40),
)
fig.show()


---
## Q2 — Which countries added the most people?

Which **15 countries added the most people** between 1970 and 2022 — and how many of them are in **Africa**?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Sorted horizontal bar chart**, coloured by **continent**. Ranking absolute people added needs a shared baseline; continent colour makes Africa’s presence in the top 15 pop visually (orange) without a separate highlight layer.

In [10]:
# Q2 — WHY a SORTED HORIZONTAL BAR chart?
# Ranking "who added the most people" needs a shared zero baseline → bars beat areas/pies.
# Horizontal layout fits long country names; sorting by gain makes the top 15 readable.
# Colour = continent so Africa's count in the top 15 is visible at a glance.

CONT = {
    'Africa': '#FF6B35', 'Asia': '#4C78A8', 'Europe': '#F2C14E',
    'North America': '#9B5DE5', 'South America': '#00BBF9', 'Oceania': '#00F5D4',
}

df['Pop_gain'] = df['2022 Population'] - df['1970 Population']
top15 = df.nlargest(15, 'Pop_gain').copy()
n_africa = int((top15['Continent'] == 'Africa').sum())
order = top15.sort_values('Pop_gain', ascending=True)['Country'].tolist()

fig = px.bar(
    top15, x='Pop_gain', y='Country', color='Continent', orientation='h',
    color_discrete_map=CONT, category_orders={'Country': order},
    labels={'Pop_gain': 'People added 1970–2022', 'Country': '', 'Continent': ''},
    template='plotly_white', height=540,
)
gain_map = top15.set_index('Country')['Pop_gain']
for tr in fig.data:
    tr.text = [f"{gain_map.loc[c]/1e6:.0f}M" for c in tr.y]
    tr.textposition = 'outside'
    tr.cliponaxis = False

fig.update_layout(
    title=f'India & China lead absolute gains — {n_africa} African countries glow orange in the top 15',
    font=dict(family='Arial', size=13),
    xaxis=dict(showgrid=True, gridcolor='#F0F0F0', title='', showticklabels=False),
    yaxis=dict(showgrid=False),
    legend=dict(orientation='h', x=0, y=1.1, title=''),
    plot_bgcolor='#FFF9F5',
    margin=dict(l=130, r=70, t=80, b=40), bargap=0.22,
)
fig.show()


---
## Q3 — Do bigger countries have more people?

Is there a relationship between a country's **land area** and its **2022 population** — does it differ by **continent**, and which countries clearly break the pattern?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Bubble scatter** (log–log): x = area, y = population, **size = density**, colour = continent. Shows the area–population link, continent clusters, and density outliers (India dense; Russia/Canada sparse) in one colourful view.

In [11]:
# Q3 — WHY a BUBBLE SCATTER (log–log)?
# Area and population are two continuous variables → scatter shows their relationship.
# Log scales handle orders-of-magnitude range; colour = continent clusters.
# Bubble size = density adds a 3rd variable so dense India vs sparse Russia/Canada stand out.

import math

CONT = {
    'Africa': '#FF6B35', 'Asia': '#4C78A8', 'Europe': '#F2C14E',
    'North America': '#9B5DE5', 'South America': '#00BBF9', 'Oceania': '#00F5D4',
}

fig = px.scatter(
    df, x='Area', y='2022 Population', color='Continent', size='Density',
    size_max=45, hover_name='Country', log_x=True, log_y=True,
    color_discrete_map=CONT,
    labels={'Area': 'Land area (km², log)', '2022 Population': '2022 population (log)',
            'Continent': '', 'Density': 'Density'},
    template='plotly_white', height=560, opacity=0.8,
)

for country, ax, ay in [
    ('China', 45, -35), ('India', 50, 30), ('Russia', -55, 35),
    ('Canada', -45, -30), ('Australia', 45, -25),
]:
    row = df[df['Country'] == country].iloc[0]
    fig.add_annotation(
        x=math.log10(row['Area']), y=math.log10(row['2022 Population']),
        text=country, showarrow=True, arrowhead=2, ax=ax, ay=ay,
        font=dict(size=12, family='Arial', color=CONT[row['Continent']]),
        arrowcolor=CONT[row['Continent']],
    )

fig.update_layout(
    title='Bubble size = density: India packs people tightly; Russia & Canada are huge but sparse',
    font=dict(family='Arial', size=12),
    legend=dict(orientation='h', x=0, y=1.1, title=''),
    xaxis=dict(showgrid=True, gridcolor='#F0F0F0'), yaxis=dict(showgrid=False),
    plot_bgcolor='#F7FFFB', margin=dict(l=60, r=30, t=90, b=50),
)
fig.show()


---
## Q4 — How crowded are the continents?

How does **population density vary within and across continents**? Show the spread, not just the average.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Colourful box plot** (one hue per continent, log density). Boxes show within-continent spread; colour doubles as a category key. Log scale keeps Macau-scale outliers readable.

In [12]:
# Q4 — WHY a BOX plot (not a bar of averages)?
# The question asks for *spread within continents*, not just a mean — boxes show median, IQR, outliers.
# Log x-axis keeps extreme city-states (e.g. Macau) visible without crushing the rest.
# One colour per continent keeps categories distinct while comparing shapes.

CONT = {
    'Africa': '#FF6B35', 'Asia': '#4C78A8', 'Europe': '#F2C14E',
    'North America': '#9B5DE5', 'South America': '#00BBF9', 'Oceania': '#00F5D4',
}

order = df.groupby('Continent')['Density'].median().sort_values(ascending=True).index.tolist()
macau = df.loc[df['Density'].idxmax()]

fig = px.box(
    df, x='Density', y='Continent', color='Continent', points='outliers',
    color_discrete_map=CONT, category_orders={'Continent': order},
    labels={'Density': 'People per km²', 'Continent': ''},
    template='plotly_white', height=500, log_x=True,
)
fig.update_traces(marker=dict(size=5, opacity=0.65), line=dict(width=2))
fig.update_layout(
    title=f"Rainbow of density: South America sparsest; {macau['Country']} ({macau['Density']:,.0f}/km²) tops Asia's tail",
    font=dict(family='Arial', size=13),
    xaxis=dict(gridcolor='#F0F0F0', title='Density (people/km², log)'),
    yaxis=dict(showgrid=False), showlegend=False,
    plot_bgcolor='#FDF7FF', margin=dict(l=120, r=40, t=70, b=50),
)
fig.show()


---
## Q5 — Where did populations grow — and where did they shrink?

Which countries' populations **grew and which shrank between 2000 and 2022** (percentage change), and where in the world are they?

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Choropleth** with a custom **magenta–teal diverging** scale centred at 0. Geography answers “where”; magenta = shrink, teal = boom. Ocean/land styling separates the map from a flat theme.

In [13]:
# Q5 — WHY a CHOROPLETH (diverging)?
# The question is geographic: *where* did populations grow or shrink → map encodes location.
# % change is above/below zero → diverging scale (magenta shrink, teal grow) centred at 0.
# ISO-3 (`CCA3`) matches Plotly's built-in country geometries.

df['pct_change'] = (df['2022 Population'] - df['2000 Population']) / df['2000 Population'] * 100

scale = [
    [0.0, '#C51B7D'], [0.25, '#E9A3C9'], [0.5, '#F7F7F7'],
    [0.75, '#5AB4AC'], [1.0, '#01665E'],
]

fig = px.choropleth(
    df, locations='CCA3', locationmode='ISO-3', color='pct_change',
    color_continuous_scale=scale, color_continuous_midpoint=0,
    range_color=[-40, 100],
    hover_name='Country',
    hover_data={'pct_change': ':.1f', 'CCA3': False, 'Continent': True},
    labels={'pct_change': '% change 2000–2022', 'Continent': 'Continent'},
    projection='natural earth', template='plotly_white', height=540,
)
fig.update_layout(
    title='Magenta = shrinking, teal = booming — Africa & the Gulf surge while Europe fades',
    font=dict(family='Arial', size=12),
    geo=dict(showframe=False, bgcolor='#F5F9FC', showocean=True, oceancolor='#D6EAF8',
             showland=True, landcolor='#EEEEEE', showcountries=True, countrycolor='#CCCCCC'),
    coloraxis_colorbar=dict(title='% change', thickness=16, len=0.65),
    margin=dict(l=0, r=0, t=60, b=0),
)
fig.show()


---
## Q6 — Where do the world's 8 billion people live?

How is the **2022 world population** distributed across **continents and the countries within them**? Show both levels in one figure.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**Treemap** coloured by **world population share** on a Turbo continuous scale (not flat continent fills). Area = headcount hierarchy; heat = global importance — China and India light up.

In [16]:
# Q6 — WHY a TREEMAP?
# Need *two hierarchy levels* (continent → country) in one figure → treemap nests both.
# Tile area = 2022 headcount (additive); colour = world share % so China/India heat up.
# Better than stacked bars here because there are many countries per continent.

fig = px.treemap(
    df, path=['Continent', 'Country'], values='2022 Population',
    color='World share (%)', color_continuous_scale='Turbo',
    template='plotly_white', height=680, width=1050,
)
fig.update_traces(
    texttemplate='<b>%{label}</b><br>%{percentEntry:.1%}',
    textfont=dict(size=13),
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} people<br>%{percentRoot:.1%} of world<br>World share: %{color:.2f}%<extra></extra>',
    marker=dict(line=dict(width=2, color='white')),
)
fig.update_layout(
    title="Turbo heat map of people: Asia blazes hot — China & India are the brightest tiles",
    font=dict(family='Arial', size=12),
    coloraxis_colorbar=dict(title='World share %', thickness=14, len=0.5),
    margin=dict(l=10, r=10, t=60, b=10),
)
fig.show()


---
## Q7 — Whose share of the world is rising?

How have the shares of world population held by **Africa, Asia and Europe** changed from 1970 to 2022? Group the other three continents as `'Rest of the world'`.

**Chart choice & justification:** *Which chart did you choose, and why? Justify design choices.

**100% stacked bars** with four vivid group colours. Shares over years are a part-to-whole story; Africa (orange) and Europe (gold) carry % labels because those two drives the narrative.

In [15]:
# Q7 — WHY a 100% STACKED BAR?
# The story is *part-to-whole shares* over years (Africa/Asia/Europe/Rest) → stacked bars.
# 100% scaling focuses on mix, not absolute totals; years on x show 1970→2022 change.
# Labels on Africa & Europe highlight the rising vs shrinking shares.

tmp = df_long.copy()
tmp['Group'] = tmp['Continent'].where(
    tmp['Continent'].isin(['Africa', 'Asia', 'Europe']), 'Rest of the world'
)
ann = tmp.groupby(['Year', 'Group'], as_index=False)['Population'].sum()
ann['Share'] = ann['Population'] / ann.groupby('Year')['Population'].transform('sum')
ann['Label'] = np.where(
    ann['Group'].isin(['Africa', 'Europe']),
    ann['Share'].map('{:.0%}'.format), ''
)

group_colors = {
    'Africa': '#FF6B35', 'Asia': '#4C78A8',
    'Europe': '#F2C14E', 'Rest of the world': '#9B5DE5',
}

fig = px.bar(
    ann, x='Year', y='Share', color='Group', text='Label',
    color_discrete_map=group_colors,
    category_orders={'Group': ['Africa', 'Asia', 'Europe', 'Rest of the world']},
    labels={'Share': 'Share of world population', 'Group': '', 'Year': ''},
    template='plotly_white', height=500,
)
fig.update_traces(textposition='inside', textfont=dict(size=12, color='white'),
                  marker_line_color='white', marker_line_width=1.5)
fig.update_layout(
    title="Orange rises, gold shrinks: Africa 10%→18%, Europe 18%→9% of world population",
    font=dict(family='Arial', size=13), barmode='stack',
    legend=dict(orientation='h', x=0, y=1.12, title=''),
    yaxis=dict(tickformat='.0%', title='', range=[0, 1], showgrid=True, gridcolor='#F0F0F0'),
    xaxis=dict(type='category', showgrid=False),
    plot_bgcolor='#FFF8F3', margin=dict(l=50, r=30, t=90, b=40),
)
fig.show()
